# Time Series Cross-Validation

A single train/test split gives one estimate of out-of-sample accuracy.
That estimate depends heavily on *where* the split happens — a lucky or
unlucky cut can make a method look better or worse than it really is.

**Time series cross-validation** gives a more robust estimate by evaluating
the model at **many different forecast origins**, simulating repeated
real-world forecasting.

## Why standard k-fold CV fails

Standard k-fold cross-validation randomly shuffles and splits data into
folds.  This destroys the temporal ordering:
- Future observations leak into training folds
- Past observations appear in test folds
- Autocorrelation structure is broken

**Result**: overly optimistic accuracy estimates that do not reflect real
forecasting performance.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import mean_absolute_error

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

In [ ]:
# Load airline passengers
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.index.freq = "MS"
airline.columns = ["Passengers"]
passengers = airline["Passengers"]

print(f"Shape: {airline.shape}")
print(f"Period: {airline.index[0].date()} to {airline.index[-1].date()}")

---
## 1. Why k-Fold CV Fails for Time Series

Let us visualise the difference between random k-fold and temporal
cross-validation.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

n = len(airline)
indices = np.arange(n)

# --- k-Fold (WRONG) ---
ax = axes[0]
np.random.seed(42)
folds = np.random.choice(5, size=n)
colors_kfold = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]
for fold in range(5):
    mask = folds == fold
    ax.scatter(indices[mask], passengers.values[mask],
              c=colors_kfold[fold], s=10, label=f"Fold {fold+1}")
ax.set_title("WRONG: k-Fold CV (random assignment) — observations are interleaved")
ax.set_ylabel("Passengers")
ax.legend(fontsize=8, ncol=5, loc="upper left")

# --- Temporal CV (RIGHT) ---
ax = axes[1]
fold_size = n // 5
for fold in range(5):
    start = fold * fold_size
    end = start + fold_size if fold < 4 else n
    ax.scatter(indices[start:end], passengers.values[start:end],
              c=colors_kfold[fold], s=10, label=f"Fold {fold+1}")
ax.set_title("RIGHT: Temporal CV — folds are contiguous time blocks")
ax.set_ylabel("Passengers")
ax.set_xlabel("Observation Index")
ax.legend(fontsize=8, ncol=5, loc="upper left")

plt.tight_layout()
plt.show()

print("In k-fold CV, each fold contains observations from all time periods.")
print("The model can 'see' the future when predicting the past — this is data leakage.")

---
## 2. Expanding Window Cross-Validation

Also called **walk-forward validation** or **time series split**.  The
training set **expands** at each iteration:

```
Fold 1: [==TRAIN==]  [TEST]
Fold 2: [===TRAIN===]  [TEST]
Fold 3: [====TRAIN====]  [TEST]
Fold 4: [=====TRAIN=====]  [TEST]
...
```

At each fold:
- Train on observations $[1, \ldots, t]$
- Forecast observations $[t+1, \ldots, t+h]$
- Compute the error
- Move $t$ forward by one step and repeat

In [ ]:
# Visualise expanding window
min_train = 60  # minimum training size (5 years)
horizon = 12    # forecast 12 months ahead
n_folds = 8     # show 8 folds for illustration
step = (len(passengers) - min_train - horizon) // (n_folds - 1)

fig, ax = plt.subplots(figsize=(14, 6))

for fold in range(n_folds):
    train_end = min_train + fold * step
    test_start = train_end
    test_end = min(test_start + horizon, len(passengers))

    y = n_folds - fold  # y-position for this fold

    # Training block
    ax.barh(y, train_end, left=0, height=0.6, color="tab:blue", alpha=0.7)
    # Test block
    ax.barh(y, test_end - test_start, left=test_start, height=0.6,
            color="tab:orange", alpha=0.7)

ax.set_xlabel("Observation Index")
ax.set_ylabel("Fold")
ax.set_yticks(range(1, n_folds + 1))
ax.set_yticklabels([f"Fold {n_folds - i}" for i in range(n_folds)])
ax.set_title("Expanding Window Cross-Validation")

# Legend
train_patch = mpatches.Patch(color="tab:blue", alpha=0.7, label="Train")
test_patch = mpatches.Patch(color="tab:orange", alpha=0.7, label="Test")
ax.legend(handles=[train_patch, test_patch], loc="lower right")

plt.tight_layout()
plt.show()

print("Key feature: the training set GROWS at each fold.")
print("All historical data is used — nothing is discarded.")

---
## 3. Implementing Expanding Window from Scratch

In [ ]:
def expanding_window_cv(series, min_train, horizon, step=1):
    """
    Generate expanding window train/test splits.

    Parameters
    ----------
    series : pd.Series
        The full time series.
    min_train : int
        Minimum number of training observations.
    horizon : int
        Forecast horizon (test set size).
    step : int
        How many observations to move forward between folds.

    Yields
    ------
    (train, test) : tuple of pd.Series
    """
    for i in range(min_train, len(series) - horizon + 1, step):
        train = series.iloc[:i]
        test = series.iloc[i:i + horizon]
        yield train, test


# Count the number of folds
folds = list(expanding_window_cv(passengers, min_train=60, horizon=12, step=1))
print(f"Number of folds (step=1): {len(folds)}")
print(f"First fold: train size={len(folds[0][0])}, test size={len(folds[0][1])}")
print(f"Last fold:  train size={len(folds[-1][0])}, test size={len(folds[-1][1])}")

In [ ]:
# Expanding window CV for the naive method
def forecast_naive(train_s, h):
    return np.full(h, train_s.iloc[-1])


def forecast_seasonal_naive(train_s, h, m=12):
    last_season = train_s.values[-m:]
    return np.tile(last_season, int(np.ceil(h / m)))[:h]


def forecast_mean(train_s, h):
    return np.full(h, train_s.mean())


def forecast_drift(train_s, h):
    slope = (train_s.iloc[-1] - train_s.iloc[0]) / (len(train_s) - 1)
    return np.array([train_s.iloc[-1] + i * slope for i in range(1, h + 1)])


# Run expanding window CV for all methods
min_train_size = 60
forecast_horizon = 12
cv_step = 1

methods = {
    "Mean": forecast_mean,
    "Naive": forecast_naive,
    "Seasonal Naive": lambda t, h: forecast_seasonal_naive(t, h, m=12),
    "Drift": forecast_drift,
}

cv_errors = {name: [] for name in methods}

for train, test in expanding_window_cv(passengers, min_train_size, forecast_horizon, cv_step):
    actual = test.values
    for name, func in methods.items():
        fc = func(train, forecast_horizon)
        err = mean_absolute_error(actual, fc)
        cv_errors[name].append(err)

print("Expanding Window CV Results (MAE):")
print(f"{'Method':20s} {'Mean MAE':>10s} {'Std MAE':>10s} {'Median MAE':>12s}")
print("-" * 54)
for name, errs in cv_errors.items():
    errs = np.array(errs)
    print(f"{name:20s} {errs.mean():10.2f} {errs.std():10.2f} {np.median(errs):12.2f}")

In [ ]:
# Visualise CV error distributions
fig, ax = plt.subplots(figsize=(12, 5))

data_to_plot = [np.array(cv_errors[name]) for name in methods]
bp = ax.boxplot(data_to_plot, labels=list(methods.keys()), patch_artist=True)

colors = ["tab:red", "tab:green", "tab:purple", "tab:brown"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)

ax.set_ylabel("MAE")
ax.set_title("Expanding Window CV — MAE Distribution by Method")
plt.tight_layout()
plt.show()

print("The box plots show the distribution of MAE across all CV folds.")
print("Lower and tighter is better.")

In [ ]:
# Plot MAE over time (by fold origin)
fold_origins = list(range(min_train_size, len(passengers) - forecast_horizon + 1, cv_step))
origin_dates = passengers.index[fold_origins]

fig, ax = plt.subplots(figsize=(14, 5))
for name, color in zip(methods.keys(), colors):
    ax.plot(origin_dates, cv_errors[name], label=name, color=color, alpha=0.7, linewidth=1)

ax.set_xlabel("Forecast Origin")
ax.set_ylabel("MAE")
ax.set_title("MAE by Forecast Origin — Expanding Window CV")
ax.legend()
plt.tight_layout()
plt.show()

print("Errors generally increase over time as the series level grows.")
print("This is expected for scale-dependent metrics on a trending series.")

---
## 4. Sliding Window Cross-Validation

In the sliding window approach, the training set has a **fixed size**.
As the window moves forward, old observations are *dropped*:

```
Fold 1:       [==TRAIN==]  [TEST]
Fold 2:        [==TRAIN==]  [TEST]
Fold 3:         [==TRAIN==]  [TEST]
Fold 4:          [==TRAIN==]  [TEST]
...
```

This is useful when **only recent data is relevant** — for example, when
the data-generating process changes over time (concept drift).

In [ ]:
# Visualise sliding window
window_size = 60
n_folds_vis = 8
step_vis = (len(passengers) - window_size - horizon) // (n_folds_vis - 1)

fig, ax = plt.subplots(figsize=(14, 6))

for fold in range(n_folds_vis):
    train_start = fold * step_vis
    train_end = train_start + window_size
    test_start = train_end
    test_end = min(test_start + horizon, len(passengers))

    y = n_folds_vis - fold

    # Unused (before window)
    if train_start > 0:
        ax.barh(y, train_start, left=0, height=0.6, color="lightgrey", alpha=0.5)

    # Training block
    ax.barh(y, window_size, left=train_start, height=0.6, color="tab:blue", alpha=0.7)
    # Test block
    ax.barh(y, test_end - test_start, left=test_start, height=0.6,
            color="tab:orange", alpha=0.7)

ax.set_xlabel("Observation Index")
ax.set_ylabel("Fold")
ax.set_yticks(range(1, n_folds_vis + 1))
ax.set_yticklabels([f"Fold {n_folds_vis - i}" for i in range(n_folds_vis)])
ax.set_title("Sliding Window Cross-Validation")

train_patch = mpatches.Patch(color="tab:blue", alpha=0.7, label="Train")
test_patch = mpatches.Patch(color="tab:orange", alpha=0.7, label="Test")
unused_patch = mpatches.Patch(color="lightgrey", alpha=0.5, label="Unused (dropped)")
ax.legend(handles=[train_patch, test_patch, unused_patch], loc="lower right")

plt.tight_layout()
plt.show()

print("Key feature: the training window has a FIXED size.")
print("Old observations are dropped as the window slides forward.")

---
## 5. Implementing Sliding Window from Scratch

In [ ]:
def sliding_window_cv(series, window_size, horizon, step=1):
    """
    Generate sliding (fixed-size) window train/test splits.

    Parameters
    ----------
    series : pd.Series
        The full time series.
    window_size : int
        Fixed training window size.
    horizon : int
        Forecast horizon (test set size).
    step : int
        How many observations to slide forward between folds.

    Yields
    ------
    (train, test) : tuple of pd.Series
    """
    for i in range(0, len(series) - window_size - horizon + 1, step):
        train = series.iloc[i:i + window_size]
        test = series.iloc[i + window_size:i + window_size + horizon]
        yield train, test


# Count folds
slide_folds = list(sliding_window_cv(passengers, window_size=60, horizon=12, step=1))
print(f"Number of sliding window folds: {len(slide_folds)}")
print(f"Each training set has exactly {len(slide_folds[0][0])} observations.")

In [ ]:
# Sliding window CV for all methods
slide_errors = {name: [] for name in methods}

for train, test in sliding_window_cv(passengers, window_size=60, horizon=12, step=1):
    actual = test.values
    for name, func in methods.items():
        fc = func(train, forecast_horizon)
        err = mean_absolute_error(actual, fc)
        slide_errors[name].append(err)

print("Sliding Window CV Results (MAE):")
print(f"{'Method':20s} {'Mean MAE':>10s} {'Std MAE':>10s} {'Median MAE':>12s}")
print("-" * 54)
for name, errs in slide_errors.items():
    errs = np.array(errs)
    print(f"{name:20s} {errs.mean():10.2f} {errs.std():10.2f} {np.median(errs):12.2f}")

---
## 6. Comparing Expanding vs Sliding Window

In [ ]:
# Side-by-side comparison
comparison = pd.DataFrame({
    "Method": list(methods.keys()),
    "Expanding (Mean MAE)": [np.mean(cv_errors[n]) for n in methods],
    "Sliding (Mean MAE)": [np.mean(slide_errors[n]) for n in methods],
})
comparison["Difference"] = comparison["Expanding (Mean MAE)"] - comparison["Sliding (Mean MAE)"]
comparison = comparison.round(2)
print(comparison.to_string(index=False))
print()
print("Negative difference means expanding window gave lower MAE.")
print("Positive difference means sliding window gave lower MAE.")

In [ ]:
# Visual comparison: expanding vs sliding for seasonal naive
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot(
    [np.array(cv_errors[n]) for n in methods],
    labels=list(methods.keys()),
    patch_artist=True,
)
axes[0].set_title("Expanding Window")
axes[0].set_ylabel("MAE")
axes[0].tick_params(axis="x", rotation=30)

axes[1].boxplot(
    [np.array(slide_errors[n]) for n in methods],
    labels=list(methods.keys()),
    patch_artist=True,
)
axes[1].set_title("Sliding Window (w=60)")
axes[1].set_ylabel("MAE")
axes[1].tick_params(axis="x", rotation=30)

plt.suptitle("Expanding vs Sliding Window CV — MAE Distributions", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Effect of Window Size

For sliding window CV, the window size is a tuning parameter.  Larger
windows use more history (better for stable processes); smaller windows
adapt faster to changes.

In [ ]:
window_sizes = [24, 36, 48, 60, 72, 84, 96]
window_results = {}

for w in window_sizes:
    errors_by_w = []
    for train, test in sliding_window_cv(passengers, window_size=w, horizon=12, step=1):
        fc = forecast_seasonal_naive(train, 12, m=12)
        err = mean_absolute_error(test.values, fc)
        errors_by_w.append(err)
    window_results[w] = np.array(errors_by_w)

fig, ax = plt.subplots(figsize=(12, 5))
means = [window_results[w].mean() for w in window_sizes]
stds = [window_results[w].std() for w in window_sizes]
ax.errorbar(window_sizes, means, yerr=stds, marker="o", capsize=5, linewidth=2)
ax.set_xlabel("Window Size (months)")
ax.set_ylabel("Mean MAE (+/- 1 std)")
ax.set_title("Effect of Window Size on Sliding Window CV — Seasonal Naive")
plt.tight_layout()
plt.show()

best_w = window_sizes[np.argmin(means)]
print(f"Best window size: {best_w} months (mean MAE = {min(means):.2f})")

---
## 8. Effect of Forecast Horizon

We can also examine how accuracy changes with the forecast horizon $h$
by collecting errors at each step ahead across all folds.

In [ ]:
# Collect per-step errors across expanding window folds
max_horizon = 12
step_errors = {name: [[] for _ in range(max_horizon)] for name in methods}

for train, test in expanding_window_cv(passengers, min_train=60, horizon=max_horizon, step=1):
    actual_vals = test.values
    for name, func in methods.items():
        fc = func(train, max_horizon)
        for h in range(max_horizon):
            step_errors[name][h].append(abs(actual_vals[h] - fc[h]))

fig, ax = plt.subplots(figsize=(12, 5))
for name, color in zip(methods.keys(), colors):
    mean_by_h = [np.mean(step_errors[name][h]) for h in range(max_horizon)]
    ax.plot(range(1, max_horizon + 1), mean_by_h, label=name, color=color,
            marker="o", markersize=5, linewidth=2)

ax.set_xlabel("Forecast Horizon (h)")
ax.set_ylabel("Mean Absolute Error")
ax.set_title("MAE by Forecast Horizon — Expanding Window CV")
ax.legend()
ax.set_xticks(range(1, max_horizon + 1))
plt.tight_layout()
plt.show()

print("Errors increase with forecast horizon for all methods.")
print("Seasonal naive has the best accuracy across all horizons.")

---
## 9. Using sklearn's TimeSeriesSplit

`sklearn.model_selection.TimeSeriesSplit` provides a built-in expanding
window splitter.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

print("sklearn TimeSeriesSplit (5 folds):")
print("=" * 50)
for fold, (train_idx, test_idx) in enumerate(tscv.split(passengers)):
    print(f"Fold {fold + 1}: train=[{train_idx[0]}:{train_idx[-1]}] ({len(train_idx)} obs), "
          f"test=[{test_idx[0]}:{test_idx[-1]}] ({len(test_idx)} obs)")

print()
print("Note: sklearn's TimeSeriesSplit is expanding window only.")
print("For sliding window, use our custom implementation.")

In [ ]:
# Using TimeSeriesSplit with a specified test size
tscv_12 = TimeSeriesSplit(n_splits=5, test_size=12)

print("TimeSeriesSplit with test_size=12:")
print("=" * 50)
for fold, (train_idx, test_idx) in enumerate(tscv_12.split(passengers)):
    print(f"Fold {fold + 1}: train size={len(train_idx)}, test size={len(test_idx)}")

---
## 10. When to Use Each Strategy

| Strategy | Training set | Best when... |
|---|---|---|
| **Expanding window** | Grows each fold | All historical data is relevant; stable process |
| **Sliding window** | Fixed size | Recent data is more relevant; concept drift; regime changes |
| **Single train/test** | Fixed | Quick evaluation; large datasets where CV is too slow |

**General advice**:
- Use **expanding window** by default — it uses all available data.
- Switch to **sliding window** if there is evidence of structural breaks
  or if the data-generating process changes over time.
- Use a **larger step size** (e.g., step=12 for monthly data) if
  computation is expensive.

In [ ]:
# Demonstrate: larger step size for efficiency
# step=12 means we move one year between folds
fast_errors = {name: [] for name in methods}

for train, test in expanding_window_cv(passengers, min_train=60, horizon=12, step=12):
    actual = test.values
    for name, func in methods.items():
        fc = func(train, 12)
        err = mean_absolute_error(actual, fc)
        fast_errors[name].append(err)

print(f"Number of folds with step=12: {len(fast_errors['Mean'])} (vs {len(cv_errors['Mean'])} with step=1)")
print()
print("Results with step=12:")
print(f"{'Method':20s} {'Mean MAE':>10s}")
print("-" * 32)
for name, errs in fast_errors.items():
    print(f"{name:20s} {np.mean(errs):10.2f}")

---
## 11. Full CV Pipeline: Putting It All Together

In [ ]:
def time_series_cv(
    series,
    forecast_func,
    min_train=60,
    horizon=12,
    step=1,
    window=None,
    error_func=mean_absolute_error,
):
    """
    Run time series cross-validation.

    Parameters
    ----------
    series : pd.Series
        Full time series.
    forecast_func : callable
        Function(train_series, horizon) -> forecast array.
    min_train : int
        Minimum training size (expanding) or ignored (sliding).
    horizon : int
        Forecast horizon.
    step : int
        Step size between folds.
    window : int or None
        If None, use expanding window. If int, use sliding window.
    error_func : callable
        Error function(actual, predicted) -> scalar.

    Returns
    -------
    np.ndarray of errors, one per fold.
    """
    errors = []

    if window is None:
        # Expanding window
        for train, test in expanding_window_cv(series, min_train, horizon, step):
            fc = forecast_func(train, horizon)
            errors.append(error_func(test.values, fc))
    else:
        # Sliding window
        for train, test in sliding_window_cv(series, window, horizon, step):
            fc = forecast_func(train, horizon)
            errors.append(error_func(test.values, fc))

    return np.array(errors)


# Example usage
expanding_errs = time_series_cv(
    passengers,
    lambda t, h: forecast_seasonal_naive(t, h, m=12),
    min_train=60,
    horizon=12,
    step=1,
    window=None,
)

sliding_errs = time_series_cv(
    passengers,
    lambda t, h: forecast_seasonal_naive(t, h, m=12),
    horizon=12,
    step=1,
    window=60,
)

print(f"Expanding window — Mean MAE: {expanding_errs.mean():.2f} ({len(expanding_errs)} folds)")
print(f"Sliding window   — Mean MAE: {sliding_errs.mean():.2f} ({len(sliding_errs)} folds)")

In [ ]:
# Apply the pipeline to a second dataset
alcohol = pd.read_csv(
    DATA_DIR / "Alcohol_Sales.csv",
    index_col="DATE",
    parse_dates=True,
)
alcohol.index.freq = "MS"
alcohol.columns = ["Sales"]
alcohol = alcohol.dropna()

print("Alcohol Sales — Cross-Validation Results")
print("=" * 50)

for name, func in methods.items():
    errs = time_series_cv(
        alcohol["Sales"],
        func,
        min_train=60,
        horizon=12,
        step=12,  # step=12 for speed
        window=None,
    )
    print(f"  {name:20s}: Mean MAE = {errs.mean():.2f}, Std = {errs.std():.2f}")

---
## Summary

- **Standard k-fold CV is invalid for time series** — it breaks temporal
  ordering and causes data leakage.
- **Expanding window** (walk-forward) CV grows the training set at each fold,
  using all available history.  This is the default choice.
- **Sliding window** CV keeps a fixed training size, discarding old data.
  Use this when recent data is more relevant or the process is non-stationary.
- The **step size** controls the number of folds — larger steps are faster
  but give fewer error estimates.
- Examining errors **by forecast horizon** and **by forecast origin** gives
  deeper insight than a single summary number.
- The `time_series_cv()` function defined here provides a unified interface
  for both expanding and sliding window strategies.

In [ ]:
print("Chapter 06 complete.")
print()
print("Key takeaways:")
print("  1. Simple benchmarks (mean, naive, seasonal naive, drift) set the bar.")
print("  2. Residual diagnostics tell you if your model captured all the signal.")
print("  3. Use MAE + MASE for accuracy; avoid MAPE when data has zeros.")
print("  4. Time series CV gives robust accuracy estimates — never use random splits.")